# 第3回：投稿テキストの加工と可視化

**このNotebookでできるようになること**

1. 日本語の文を単語に分解する（形態素解析）
2. 頻出語・ワードクラウドで「何が話題か」を見る
3. 投稿数・反応数を時間軸で見る
4. 語と語のつながり（共起ネットワーク）を描く
5. 2つの話題を比べて「特徴的な語」を見つける（TF-IDF）

第2回で保存した `data/posts.csv` を使います．無い場合は `sample_data/posts_sample.csv`（授業用の合成データ．実在の投稿ではありません）が自動で読み込まれます．

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bsky_utils import *

pd.set_option("display.max_colwidth", 60)
set_japanese_font()

df = load_posts("data/posts.csv")
print(df["query"].value_counts())
df[["query", "created_at_jst", "text", "like_count"]].head()

## 1. 形態素解析：文を単語に分ける

英語と違って日本語は単語の区切りに空白がありません．**形態素解析**は文を単語（形態素）に分けて品詞を付ける処理です．ここでは純Pythonで動く `janome` を使います．

`tokenize(文)` は，URLやメンションを取り除いたうえで，単語を**原形**（「行わ」→「行う」）にして返します．

In [ ]:
# 品詞付きで見てみる（janomeを直接使う）
from janome.tokenizer import Tokenizer
t = Tokenizer()
for tok in t.tokenize("避難所の案内が分かりにくい。看板をもっと増やしてほしい。"):
    print(f"{tok.surface:8s} {tok.part_of_speech:20s} 原形={tok.base_form}")

### 前処理：意味の薄い語を落とす（切り替え可能）

形態素解析の結果をそのまま数えると，「する」「てる」「れる」「ない」「ください」のような **どんな文にも出る語** が上位を占めて，話題が見えません．
`tokenize()` は次の4段階で語を選びます．どの段階も **オプションで切り替え** られます．

| 段階 | 何をするか | オプション |
|---|---|---|
| 1. 品詞で選ぶ | 名詞・動詞・形容詞だけ残す（助詞・助動詞・記号は落とす） | `pos=("名詞",)` など |
| 2. 品詞の細分類で落とす | 補助動詞「てる・ください・すぎる」，接尾「れる・られる」，非自立形容詞「ない・ほしい・にくい」，数・代名詞・接尾名詞 | `exclude_detail=set()` で無効 |
| 3. 短いひらがな語を落とす | 2文字以下でひらがなだけの語（「ん」「もの」など） | `max_hiragana_len=0` で無効 |
| 4. ストップワード | `stopwords_ja.txt`（同じフォルダ，1行1語）＋ 追加語 | `extra_stopwords={...}` / `stopwords=set()` |

よく使う組み合わせは **プリセット** として用意しています．

| preset | 残す語 | 向いている場面 |
|---|---|---|
| `"content"`（既定） | 名詞・動詞・形容詞から意味の薄い語を除いたもの | 頻出語，TF-IDF，分類 |
| `"nouns"` | 名詞だけ | 「何について」語られているか（話題） |
| `"nouns_adj"` | 名詞＋形容詞 | 評価・感想の語も見たいとき |
| `"raw"` | ほぼ全部（副詞・感動詞も） | 前処理の効果を比べるとき |

In [ ]:
sample_text = "大雨警報が発表されてる。警戒してください。避難所の案内が分かりにくい。看板をもっと増やしてほしい。"
for preset in ["raw", "content", "nouns", "nouns_adj"]:
    print(f"{preset:10s}", tokenize(sample_text, preset=preset))

`explain_tokens()` で，1文の各語が **残ったか／なぜ落ちたか** を確認できます．前処理を変えたときは，まずこれで挙動を確かめましょう．

In [ ]:
explain_tokens("避難所の案内が分かりにくい。看板をもっと増やしてほしい。")

In [ ]:
# 個別に切り替える例
print("細分類の除外を無効に:", tokenize(sample_text, exclude_detail=set()))
print("ひらがな規則を無効に:", tokenize(sample_text, max_hiragana_len=0))
print("ストップワードを追加:", tokenize(sample_text, extra_stopwords={"警報", "発表"}))
print("動詞だけ:", tokenize(sample_text, pos=("動詞",), exclude_detail=set(), max_hiragana_len=0, stopwords=set()))

### ストップワードの編集

- 恒久的に落としたい語は **`stopwords_ja.txt`**（このフォルダ，1行1語，`#` はコメント）に追記します．Notebook を再起動すると反映されます（`load_stopwords()` で読み直すこともできます）．
- その分析だけで落としたい語（**検索語そのもの**など）は `extra_stopwords={...}` で渡します．検索語は必ず出るので情報になりません．

In [ ]:
print("ストップワードの数:", len(DEFAULT_STOPWORDS), " ファイル:", STOPWORDS_PATH.name)
print(sorted(DEFAULT_STOPWORDS)[:30], "...")

## 2. 全投稿を単語にして，頻出語を数える

検索語（`query`）そのものはストップワードに加えます．

In [ ]:
extra = set(df["query"].unique())               # 検索語そのものは落とす
df["words"] = df["text"].map(lambda s: tokenize(s, extra_stopwords=extra))    # 各投稿 → 単語のリスト
df[["text", "words"]].head()

In [ ]:
counter = Counter(w for ws in df["words"] for w in ws)
top30 = counter.most_common(30)
top30

In [ ]:
labels, values = zip(*top30)
plt.figure(figsize=(8, 8))
plt.barh(labels[::-1], values[::-1])
plt.xlabel("出現回数")
plt.title("頻出語 Top30（全投稿）")
plt.tight_layout()
plt.show()

### 前処理を変えると上位語はどう変わるか

同じデータを `raw` / `content` / `nouns` で処理して上位語を並べます．**前処理の選択が結果を決める**ことを確認してください．

In [ ]:
compare = {}
for preset in ["raw", "content", "nouns"]:
    c = Counter(w for text in df["text"] for w in tokenize(text, preset=preset, extra_stopwords=extra))
    compare[preset] = [w for w, _ in c.most_common(15)]
pd.DataFrame(compare)

それでも残る「ニュース」「情報」のような語を落としたければ，`extra_stopwords` に加えるか `stopwords_ja.txt` に追記します．

In [ ]:
extra = set(df["query"].unique()) | {"ニュース", "情報"}      # ← 自由に追加
df["words"] = df["text"].map(lambda s: tokenize(s, extra_stopwords=extra))
counter = Counter(w for ws in df["words"] for w in ws)
counter.most_common(20)

## 3. ワードクラウド

頻出語を「大きさ」で表す図です．発表のつかみには便利ですが，**正確な比較には向かない**（棒グラフを併用する）ことを覚えておきましょう．

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(font_path=japanese_font_path(), width=900, height=500,
               background_color="white", colormap="viridis", max_words=100)
wc.generate_from_frequencies(counter)

plt.figure(figsize=(10, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("ワードクラウド（全投稿）")
plt.show()

## 4. 時間軸で見る：投稿数の推移

「いつ盛り上がったか」は，ニュースや出来事と対応づけて解釈できます．

In [ ]:
daily = df.groupby(["date", "query"]).size().unstack(fill_value=0)
daily.plot(figsize=(10, 4), marker="o")
plt.ylabel("投稿数")
plt.title("日別の投稿数")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 曜日 × 時間帯 のヒートマップ
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = df.pivot_table(index="weekday", columns="hour", values="post_id", aggfunc="count", fill_value=0)
pivot = pivot.reindex([d for d in order if d in pivot.index])

plt.figure(figsize=(12, 3.5))
plt.imshow(pivot.values, aspect="auto", cmap="Blues")
plt.colorbar(label="投稿数")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.xlabel("時刻（時）")
plt.title("曜日×時間帯の投稿数")
plt.tight_layout()
plt.show()

### 特定の語の出現数の推移

「避難」「警報」などの語が **いつ増えたか** を追うと，出来事との関係が見えます．

In [ ]:
watch_words = [w for w, _ in counter.most_common(5)]      # ← 追いたい語のリストに書き換えてよい

for w in watch_words:
    df[f"has_{w}"] = df["words"].map(lambda ws: w in ws)

trend = df.groupby("date")[[f"has_{w}" for w in watch_words]].sum()
trend.columns = watch_words
trend.plot(figsize=(10, 4), marker="o")
plt.ylabel("出現した投稿数")
plt.title("語ごとの出現数の推移")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 反応（いいね・リポスト）を見る

反応数は **極端に偏った分布**（ほとんどが0，ごく一部が非常に多い）になるのが普通です．平均値だけで語らず，分布を見ましょう．

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["like_count"], bins=30)
axes[0].set_title("いいね数の分布")
axes[0].set_xlabel("いいね数")
axes[1].hist(np.log10(df["like_count"] + 1), bins=30)
axes[1].set_title("log10(いいね数 + 1) の分布")
axes[1].set_xlabel("log10(いいね数+1)")
plt.tight_layout()
plt.show()

print("いいね数の中央値:", df["like_count"].median(), " 平均:", round(df["like_count"].mean(), 1))

In [ ]:
# 画像あり／なしで反応は違う？（中央値と平均を比べる）
df["has_image"] = df["image_count"] > 0
df.groupby("has_image")["like_count"].agg(["count", "median", "mean"]).round(1)

In [ ]:
# 本文の長さと反応
df["text_len"] = df["text"].str.len()
plt.figure(figsize=(6, 4))
plt.scatter(df["text_len"], df["like_count"] + 1, alpha=0.4, s=12)
plt.yscale("log")
plt.xlabel("本文の文字数")
plt.ylabel("いいね数 + 1（対数）")
plt.title("本文の長さといいね数")
plt.tight_layout()
plt.show()

> **注意**：「画像付きの方がいいねが多い」ように見えても，それは *画像のおかげ* とは限りません（人気アカウントほど画像を付ける，話題が違う，など別の理由がありえます）．**相関は因果ではない**ことを発表でも一言添えましょう．

## 6. 共起ネットワーク：どの語とどの語が一緒に使われるか

同じ投稿に出てきた語のペアを数え，回数の多いペアを線でつなぎます．「線がある＝一緒に語られている」であって，因果関係ではありません．

In [ ]:
from itertools import combinations
import networkx as nx

pair_counter = Counter()
for ws in df["words"]:
    for a, b in combinations(sorted(set(ws)), 2):
        pair_counter[(a, b)] += 1

MIN_COUNT = 3                                  # ← 線を引く最低回数（データが少なければ2に）
edges = [(a, b, c) for (a, b), c in pair_counter.most_common(60) if c >= MIN_COUNT]
print("ペア数:", len(edges))
edges[:10]

In [ ]:
G = nx.Graph()
for a, b, c in edges:
    G.add_edge(a, b, weight=c)

if G.number_of_edges():
    plt.figure(figsize=(11, 9))
    pos = nx.spring_layout(G, k=0.8, seed=42)
    deg = dict(G.degree())
    nx.draw_networkx_nodes(G, pos, node_size=[300 + 150 * deg[n] for n in G.nodes()], node_color="#cde6f7")
    nx.draw_networkx_edges(G, pos, width=[0.5 + 0.4 * G[u][v]["weight"] for u, v in G.edges()], alpha=0.5)
    nx.draw_networkx_labels(G, pos, font_size=11, font_family=plt.rcParams["font.family"][0])
    plt.title("語の共起ネットワーク")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("ペアがありません．MIN_COUNT を下げるか，データを増やしてください．")

## 7. 2つの話題を比べる：TF-IDFで「特徴語」を探す

頻出語は「よく出る語」ですが，「その話題らしい語」とは限りません．
**TF-IDF** は，ある文書群でよく出て，他の文書群ではあまり出ない語に高い値を付けます．
検索語（`query`）ごとに投稿をまとめて比較します．

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 比較するグループ：検索語が2つ以上あれば query，1つだけなら 画像あり/なし で比べる
group_col = "query" if df["query"].nunique() >= 2 else "has_image"
groups = df.groupby(group_col)["words"].apply(lambda s: [w for ws in s for w in ws])
print("比較するグループ:", list(groups.index))

vec = TfidfVectorizer(analyzer=lambda ws: ws)        # すでに単語リストなので分割しない
X = vec.fit_transform(groups.values)
terms = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 5))
axes = np.atleast_1d(axes)
for ax, (name, row) in zip(axes, zip(groups.index, X.toarray())):
    idx = row.argsort()[::-1][:12]
    ax.barh(terms[idx][::-1], row[idx][::-1])
    ax.set_title(f"{name} の特徴語（TF-IDF）")
plt.tight_layout()
plt.show()

## 8. 単語付きデータを保存する

第4回（機械学習）では，この `words` 列を使います．リストはCSVに入れにくいので空白区切りの文字列にして保存します．

In [ ]:
df["words_str"] = df["words"].map(" ".join)
save_posts(df.drop(columns=["words"]), "data/posts_words.csv", note="第3回：形態素解析済み")

## まとめ：プロジェクトで使うときのヒント

| 知りたいこと | 使う図・手法 |
|---|---|
| 何が話題か | 頻出語の棒グラフ，ワードクラウド（`preset` とストップワードを調整し，`explain_tokens` で確認） |
| いつ盛り上がったか | 日別・時間帯別の投稿数，語ごとの推移 |
| どんな文脈で語られているか | 共起ネットワーク |
| 2つの話題・時期・発信者の違い | TF-IDFの特徴語，グループ別の頻出語 |
| どんな投稿が反応されるか | 反応数の分布，画像あり/なしの比較（因果ではないことに注意） |

**次回（第4回）**：機械学習で投稿を「分類」します．自分たちでラベルを付けた少量のデータから分類器を作り，大量の投稿に適用する方法と，画像に何が写っているかを深層学習（YOLO）で調べる方法を扱います．